# Day 4 — Middleware & Dependency Injection

---

Real APIs share a lot of plumbing: pagination, auth checks, DB sessions, request timing, CORS. We don't want to copy that into every route.

FastAPI gives us two tools for this:

- **Dependency Injection (DI)** — reusable pieces of logic that *run for a route* (e.g. parse pagination, fetch the current user).
- **Middleware** — code that runs on **every** request, around the whole app (e.g. add a timing header, handle CORS).


In [ ]:
!pip install fastapi uvicorn httpx


## What is Dependency Injection?

A **dependency** is just a function (or class) that FastAPI runs for you before your route. Whatever it returns is passed in as an argument.

You declare it with `Depends(...)`.

```python
from fastapi import Depends

def my_dep():
    return "hello"

@app.get("/")
def root(value: str = Depends(my_dep)):
    return {"value": value}
```


In [3]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app = FastAPI()

def common_params(skip: int = 0, limit: int = 11):
    return {"skip": skip, "limit": limit}

@app.get("/items")
def list_items(params: dict = Depends(common_params)):
    return {"params": params, "items": [f"item-{i}" for i in range(params["skip"], params["skip"] + params["limit"])]}

@app.get("/orders")
def list_orders(params: dict = Depends(common_params)):
    return {"params": params, "items": [f"item-{i}" for i in range(params["skip"], params["skip"] + params["limit"])]}

@app.get("/cartitems")
def list_cartitems(params: dict = Depends(common_params)):
    return {"params": params, "items": [f"item-{i}" for i in range(params["skip"], params["skip"] + params["limit"])]}

client = TestClient(app)
print(client.get("/items").json())
print(client.get("/items?skip=5&limit=3").json())


{'params': {'skip': 0, 'limit': 11}, 'items': ['item-0', 'item-1', 'item-2', 'item-3', 'item-4', 'item-5', 'item-6', 'item-7', 'item-8', 'item-9', 'item-10']}
{'params': {'skip': 5, 'limit': 3}, 'items': ['item-5', 'item-6', 'item-7']}


### Why is this useful?

- **Reusable**: the same `common_params` dep can be plugged into many routes.
- **Validated**: FastAPI parses query params (`skip: int`) and returns 422 on bad input automatically.
- **Documented**: Swagger UI shows `skip` and `limit` as query params on every route that uses it.


## Common Dependency Use Cases

| Use case | What the dep returns |
|----------|----------------------|
| **Pagination** | `{"skip": int, "limit": int}` |
| **DB session** | An open session (closes on teardown) |
| **Current user** | A user object from a token/header |
| **Config / settings** | A shared `Settings` instance |
| **API client** | A pre-configured HTTP client |


## Sub-Dependencies

Dependencies can depend on other dependencies. FastAPI resolves the whole tree for you.


In [2]:
from fastapi import Header, HTTPException

app = FastAPI()

def get_token(x_token: str = Header(default="")):
    if not x_token:
        raise HTTPException(401, "missing X-Token")
    return x_token

def get_current_user(token: str = Depends(get_token)):
    # In real life: decode token, lookup user. Here, fake it.
    return {"username": f"user-of-{token}"}

@app.get("/me")
def me(user: dict = Depends(get_current_user)):
    return user

client = TestClient(app)
print(client.get("/me").status_code)                                 # 401
print(client.get("/me", headers={"X-Token": "abc"}).json())          # uses token


401
{'username': 'user-of-abc'}


## Yield-Based Dependencies (Setup/Teardown)

For resources that need cleanup (DB sessions, file handles), use a generator with `yield`.

Code **before** `yield` runs at request start. Code **after** runs when the request is done — even if an exception occurred.


In [4]:
app = FastAPI()

def get_db():
    db = {"connected": True, "rows": [{"id": 1, "name": "alice"}]}
    print(">> opening db")
    try:
        yield db
    finally:
        print("<< closing db")

@app.get("/users")
def list_users(db: dict = Depends(get_db)):
    return db["rows"]

client = TestClient(app)
print(client.get("/users").json())


>> opening db
<< closing db
[{'id': 1, 'name': 'alice'}]


## What is Middleware?

**Middleware** sits between the network and your routes. It runs on **every** request, in this shape:

```
request → [middleware 1] → [middleware 2] → route → [middleware 2 after] → [middleware 1 after] → response
```

Use middleware for cross-cutting concerns: logging, timing, CORS, compression, auth at the edge.


## Custom Middleware — Timing Header

We'll add an `X-Process-Time` header to every response showing how long the request took.


In [25]:
import time
from fastapi import Request

app = FastAPI()

@app.middleware("http")
async def add_process_time(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    response.headers["X-Process-Time"] = f"{time.perf_counter() - start:.4f}"
    return response

@app.get("/")
def root():
    return {"ok": True}

client = TestClient(app)
r = client.get("/")
print("body:", r.json())
print("X-Process-Time:", r.headers["X-Process-Time"])


body: {'ok': True}
X-Process-Time: 0.0013


### What's happening

- `call_next(request)` runs the rest of the stack (other middleware + your route).
- Anything you do **before** `call_next` runs on the way in.
- Anything **after** runs on the way out — here, we attach a header.


## Built-in Middleware: CORS

Browsers block JS calls to a different origin unless the server explicitly allows it via **CORS** headers. FastAPI ships with `CORSMiddleware`.


In [5]:
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"ok": True}

client = TestClient(app)
r = client.get("/", headers={"Origin": "http://localhost:3000"})
print("access-control-allow-origin:", r.headers.get("access-control-allow-origin"))


access-control-allow-origin: *


## Middleware vs. Dependencies — When to Use Which

| | Middleware | Dependency |
|---|---|---|
| **Scope** | Every request | Only routes that declare it |
| **Access to route info** | Limited (no parsed body) | Full (params, body, other deps) |
| **Good for** | Logging, CORS, timing, compression | Auth, DB sessions, pagination, validation |
| **Order** | Outer layer | Runs inside the route stack |


## Putting It Together

A single app combining pagination dep, current-user dep, and timing middleware.


In [6]:
import time
from fastapi import FastAPI, Depends, Header, HTTPException, Request
from fastapi.testclient import TestClient

app = FastAPI()

@app.middleware("http")
async def timing(request: Request, call_next):
    start = time.perf_counter()
    resp = await call_next(request)
    resp.headers["X-Process-Time"] = f"{time.perf_counter() - start:.4f}"
    return resp

def common_pagination(skip: int = 0, limit: int = 10) -> dict:
    return {"skip": skip, "limit": limit}

def current_user(x_user: str = Header(default="")) -> str:
    if not x_user:
        raise HTTPException(401, "missing X-User header")
    return x_user

@app.get("/items")
def items(p: dict = Depends(common_pagination)):
    return {"page": p}

@app.get("/me")
def me(user: str = Depends(current_user)):
    return {"user": user}

client = TestClient(app)
print(client.get("/items?skip=2&limit=5").json())
print(client.get("/me").status_code)
print(client.get("/me", headers={"X-User": "alice"}).json())
print("timing header on /items:", client.get("/items").headers["X-Process-Time"])


{'page': {'skip': 2, 'limit': 5}}
401
{'user': 'alice'}
timing header on /items: 0.0006


## Quick Recap

- **Dependencies** are reusable functions plugged in with `Depends(...)`. Use them for pagination, auth, DB sessions.
- **Sub-dependencies** let deps depend on other deps — FastAPI resolves the tree.
- **Yield-based deps** are for setup/teardown (open + close).
- **Middleware** runs on every request — great for timing, logging, CORS.
- Use `app.add_middleware(CORSMiddleware, ...)` for cross-origin browser calls.
- Next: auth with API keys and JWT.
